In [ ]:
--train test split



-- Rows with enough history for modeling (all lags + rolling non-null)
CREATE OR REPLACE VIEW SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_MODELING_READY AS
SELECT *
FROM SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_MODELING_DATASET
WHERE CRIME_LAG_1 IS NOT NULL
  AND CRIME_LAG_2 IS NOT NULL
  AND CRIME_LAG_3 IS NOT NULL
  AND CRIME_LAG_4 IS NOT NULL
  AND ROLLING_4W IS NOT NULL;

-- Split point: 80% train / 20% test by time
SET split_cutoff = (
  SELECT DATEADD('day', -FLOOR(DATEDIFF('day', MIN(WEEK_START), MAX(WEEK_START)) * 0.2), MAX(WEEK_START))
  FROM SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_MODELING_READY
);

-- Train: older 80% of weeks
CREATE OR REPLACE VIEW SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_TRAIN AS
SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_MODELING_READY
WHERE WEEK_START < ($split_cutoff);

-- Test: latest 20% of weeks
CREATE OR REPLACE VIEW SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_TEST AS
SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_MODELING_READY
WHERE WEEK_START >= ($split_cutoff);

In [ ]:
# 1. Session (run first)
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("ALTER SESSION SET DATE_OUTPUT_FORMAT = 'YYYY-MM-DD'").collect()
session.sql("ALTER SESSION SET DATE_INPUT_FORMAT = 'YYYY-MM-DD'").collect()
print("Session ready.")

In [ ]:
-- 2. Create fetch views (run once; avoids DATE fetch error)
CREATE OR REPLACE VIEW SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_TRAIN_FETCH AS
SELECT
    DISTRICT,
    CRIME_LAG_1,
    CRIME_LAG_2,
    CRIME_LAG_3,
    CRIME_LAG_4,
    ROLLING_4W,
    WEEK_OF_YEAR,
    TARGET,
    TO_VARCHAR(WEEK_START, 'YYYY-MM-DD') AS WEEK_START
FROM SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_TRAIN;

CREATE OR REPLACE VIEW SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_TEST_FETCH AS
SELECT
    DISTRICT,
    CRIME_LAG_1,
    CRIME_LAG_2,
    CRIME_LAG_3,
    CRIME_LAG_4,
    ROLLING_4W,
    WEEK_OF_YEAR,
    TARGET,
    TO_VARCHAR(WEEK_START, 'YYYY-MM-DD') AS WEEK_START
FROM SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_TEST;

In [ ]:
# 3. Load train/test from fetch views (WEEK_START → datetime)
import pandas as pd

train_df = pd.DataFrame([r.as_dict() for r in session.sql("SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_TRAIN_FETCH").collect()])
test_df = pd.DataFrame([r.as_dict() for r in session.sql("SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_TEST_FETCH").collect()])

train_df["WEEK_START"] = pd.to_datetime(train_df["WEEK_START"], format="%Y-%m-%d")
test_df["WEEK_START"] = pd.to_datetime(test_df["WEEK_START"], format="%Y-%m-%d")

print("Train:", len(train_df), "| Test:", len(test_df))
print("Cols:", list(train_df.columns))

In [ ]:
# 4. Model + evaluate (sklearn)
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

features = [c for c in ["DISTRICT", "CRIME_LAG_1", "CRIME_LAG_2", "CRIME_LAG_3", "CRIME_LAG_4", "ROLLING_4W", "WEEK_OF_YEAR"] if c in train_df.columns]
target = "TARGET" if "TARGET" in train_df.columns else "CRIME_COUNT"

t = train_df[features + [target]].dropna()
v = test_df[features + [target]].dropna()

model = GradientBoostingRegressor(n_estimators=50, max_depth=4, random_state=42)
model.fit(t[features], t[target])
pred = model.predict(v[features])

mae = mean_absolute_error(v[target], pred)
rmse = np.sqrt(mean_squared_error(v[target], pred))
r2 = r2_score(v[target], pred)
# MAPE: exclude actual=0 to avoid div-by-zero
mask = v[target] > 0
mape = np.mean(np.abs((v.loc[mask, target] - pred[mask]) / v.loc[mask, target])) * 100 if mask.any() else float("nan")

print("Features:", features)
print("Train size:", len(t), "| Test size:", len(v))
print("MAE:", round(mae, 2), "| RMSE:", round(rmse, 2))
print("MAPE:", round(mape, 1), "%  (avg % error vs actual)")
print("R²:", round(r2, 3), "  (share of variance explained, 0–1)")

In [ ]:
# 5. Predict next week: how many crimes, which districts (run after cell 4)
import pandas as pd

# Latest week in our data (use test — most recent)
full = pd.concat([train_df, test_df], ignore_index=True)
latest_week = full["WEEK_START"].max()
latest = full[full["WEEK_START"] == latest_week].copy()

# Build "next week" inputs: shift lags (this week's TARGET → next week's LAG_1, etc.)
forecast = latest.assign(
    CRIME_LAG_1=latest["TARGET"],
    CRIME_LAG_2=latest["CRIME_LAG_1"],
    CRIME_LAG_3=latest["CRIME_LAG_2"],
    CRIME_LAG_4=latest["CRIME_LAG_3"],
    ROLLING_4W=(latest["TARGET"] + latest["CRIME_LAG_1"] + latest["CRIME_LAG_2"] + latest["CRIME_LAG_3"]) / 4,
    WEEK_OF_YEAR=(latest["WEEK_START"] + pd.Timedelta(days=7)).dt.isocalendar().week.astype(int),
)
forecast_week = latest_week + pd.Timedelta(days=7)

pred_next = model.predict(forecast[features])
out = forecast[["DISTRICT"]].assign(FORECAST_WEEK=forecast_week, PREDICTED_CRIMES=pred_next)
out = out.sort_values("PREDICTED_CRIMES", ascending=False).reset_index(drop=True)

print("Predicted crimes next week (by district, highest first):")
print(out.to_string(index=False))

In [ ]:
# 6. Save forecast to table (for Streamlit dashboard)
sf_out = session.create_dataframe(out)
sf_out.write.save_as_table("SNOWFLAKE_LEARNING_DB.PUBLIC.CRIME_FORECASTS_NEXT_WEEK", mode="overwrite")
print("Saved to CRIME_FORECASTS_NEXT_WEEK.")